# Comparing Strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/02_comparing_strategies.ipynb)

**For:** researchers who have run a first simulation, as in
[01_getting_started.ipynb](01_getting_started.ipynb). **Level:** intermediate.

You will compare strategies the way a simulation study does: on one item bank and one population,
with every design run on the same seed, and each difference judged against its Monte Carlo error.
You will compare four estimators, three selectors, four kinds of exposure control and two
variable-length stoppers, read the metrics that tell them apart (RMSE, bias, test length, exposure
and overlap), and check a run's Monte Carlo errors against replications. The notebook shows one
example of each kind of method; the
[technique pages](https://douglasrizzo.com.br/catsim/api/techniques/index.html) list them all.

To run the notebook outside this repository, install catsim with its plot extra, which adds
matplotlib: `pip install 'catsim[plot]'`.

In [ ]:
import matplotlib.pyplot as plt
import numpy

from catsim import CatEngine, ItemBank, SimulationResult, metrics, plot, simulate
from catsim.estimation import (
  BaseEstimator,
  EAPEstimator,
  MAPEstimator,
  NumericalSearchEstimator,
  WarmLikelihoodEstimator,
)
from catsim.initialization import FixedPointInitializer
from catsim.selection import (
  AStratSelector,
  BaseSelector,
  KLSelector,
  MaxInfoSelector,
  ProgressiveSelector,
  RandomesqueSelector,
  UrrySelector,
)
from catsim.stopping import BaseStopper, ConfidenceIntervalStopper, MaxLengthStopper, StandardErrorStopper, StopRule

## One bank, one population, one seed

Every design below meets the same `N_ITEMS`-item bank and the same `N_EXAMINEES` examinees, and runs
on the same seed, `RUN_SEED`. Before its test, each examinee draws an answer to every item of the
bank from their own random stream, so with the same seed an examinee answers each item the same way
in every design. The designs differ only in what they do with those answers, and comparing them
examinee by examinee removes much of the noise: these are common random numbers
([Reproducibility](https://douglasrizzo.com.br/catsim/reproducibility.html)). `BANK_SEED` and
`POPULATION_SEED` draw the bank and the true abilities. Tests have `TEST_LENGTH` items until the
section on stoppers: short tests make the strategies differ more.

`run` builds an engine that starts every test at 0, from the strategies it is given or the defaults,
and simulates it. `compare` prints, for each design, its RMSE with its Monte Carlo error, its bias,
mean test length, highest exposure rate, share of the bank never given and overlap rate. Its
`MSE - first` column is the design's mean squared error minus the first design's, computed examinee
by examinee: `metrics.mean` gives the mean of those differences with its Monte Carlo error. A
difference more than about two of its errors from 0 is unlikely to be noise.

In [ ]:
BANK_SEED = 1
POPULATION_SEED = 2
RUN_SEED = 3
N_ITEMS = 300
N_EXAMINEES = 500
TEST_LENGTH = 10

item_bank = ItemBank.generate(N_ITEMS, model="3PL", rng=BANK_SEED)
true_thetas = numpy.random.default_rng(POPULATION_SEED).normal(0, 1, N_EXAMINEES)


def run(
  *,
  selector: BaseSelector | None = None,
  estimator: BaseEstimator | None = None,
  stopper: BaseStopper | None = None,
  rng: int = RUN_SEED,
  parts: int = 1,
  workers: int = 1,
) -> SimulationResult:
  """Simulate the population on an engine that starts at 0; by default max info, EAP and TEST_LENGTH items."""
  engine = CatEngine(
    item_bank,
    FixedPointInitializer(0.0),
    selector or MaxInfoSelector(),
    estimator or EAPEstimator(),
    stopper or MaxLengthStopper(TEST_LENGTH),
  )
  return simulate(engine, true_thetas=true_thetas, rng=rng, parts=parts, workers=workers)


def squared_errors(result: SimulationResult) -> numpy.ndarray:
  """Return each examinee's squared error, (final estimate - true ability) ** 2."""
  examinees = result.examinees()
  return (examinees["theta"] - examinees["true_theta"]) ** 2


def compare(results: dict[str, SimulationResult]) -> None:
  """Print each design's metrics, and its MSE minus the first design's, paired by examinee."""
  first = squared_errors(next(iter(results.values())))
  print(f"{'':14} {'RMSE':>13} {'MSE - first':>16} {'bias':>7} {'length':>6} {'top rate':>8} {'unused':>6} overlap")
  for name, result in results.items():
    rmse = result.rmse()
    difference = metrics.mean(squared_errors(result) - first)
    rates = result.exposure_rates().value
    overlap = result.overlap_rate()
    print(
      f"{name:14} {rmse.value:.3f} ± {rmse.mc_se:.3f} {difference.value:+.3f} ± {difference.mc_se:.3f}"
      f" {result.bias().value:+7.3f} {result.test_length().value:6.1f} {rates.max():8.2f}"
      f" {numpy.mean(rates == 0):6.2f} {'None' if overlap is None else f'{overlap.value:.2f}'}"
    )

## Estimators

Four estimators, each with maximum-information selection: expected a posteriori
([EAP](https://douglasrizzo.com.br/catsim/api/techniques/estimation/eap-estimator.html)) and maximum
a posteriori
([MAP](https://douglasrizzo.com.br/catsim/api/techniques/estimation/map-estimator.html)), which
combine the responses with a standard normal prior; maximum likelihood
([ML](https://douglasrizzo.com.br/catsim/api/techniques/estimation/numerical-search-estimator.html)),
which uses the responses alone; and Warm's weighted likelihood
([WLE](https://douglasrizzo.com.br/catsim/api/techniques/estimation/warm-wle.html)), which corrects
the bias of maximum likelihood. The last line is MAP's difference from EAP computed from their two
MSEs alone, as if the runs had drawn independent random numbers.

In [ ]:
by_estimator = {
  "EAP": run(),
  "ML": run(estimator=NumericalSearchEstimator()),
  "MAP": run(estimator=MAPEstimator()),
  "WLE": run(estimator=WarmLikelihoodEstimator()),
}
compare(by_estimator)

eap, map_ = by_estimator["EAP"].mse(), by_estimator["MAP"].mse()
print(f"MAP - EAP from the two MSEs: {map_.value - eap.value:+.3f} ± {numpy.hypot(eap.mc_se, map_.mc_se):.3f}")

EAP and MAP cannot be told apart: MAP's MSE is 0.007 ± 0.011 above EAP's, within one error of 0.
Computed from the two MSEs alone, the same difference has an error of 0.030, almost three times as
large: the common random numbers make the comparison that much sharper. ML's MSE is 0.184 ± 0.050
above EAP's and WLE's 0.074 ± 0.027 above, each more than two errors from 0. The bias is close to 0
for all four, which hides how each estimator errs at the ends of the scale; the figure shows it.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharex=True, sharey=True)
for ax, (name, result) in zip(axes, by_estimator.items(), strict=True):
  examinees = result.examinees()
  ax.scatter(examinees["true_theta"], examinees["theta"], s=8, alpha=0.5)
  ax.axline((0, 0), slope=1, color="gray", linestyle="--")
  ax.set_title(name)
  ax.set_xlabel("true ability θ")
axes[0].set_ylabel("final estimate θ̂")
plt.tight_layout()
plt.show()

ml = by_estimator["ML"]
lowest = int(numpy.argmin(ml.examinees()["theta"]))
session = ml.sessions[lowest]
print(
  f"ML's lowest estimate: {session.current_theta:.2f}, true ability {ml.true_thetas[lowest]:.2f},"
  f" responses {[int(response) for response in session.responses]}"
)

Each panel plots the final estimates against the true abilities. The EAP and MAP clouds are flatter
than the diagonal: both estimators underestimate high abilities and overestimate low ones, pulled
toward the mean of their prior, and their errors at the two ends cancel in the bias. WLE follows the
diagonal at both ends but scatters more around it: less bias, more variance, and in tests this short
a larger MSE. ML scatters most, and one examinee ends at −6, the bottom of the ability scale,
although three of their ten responses are correct (the line under the figure): the guessing
parameter gives a correct response a probability of at least $c$ at any ability, so when the correct
responses look like guesses, the likelihood keeps rising toward the bottom of the scale. The
[estimation](https://douglasrizzo.com.br/catsim/api/techniques/estimation/index.html) page explains
estimates at the edges of the scale.

## Selectors

Three selectors, each with the EAP estimator: maximum information at the current estimate
([max info](https://douglasrizzo.com.br/catsim/api/techniques/selection/max-info-selector.html));
the item whose difficulty is closest to the estimate
([Urry](https://douglasrizzo.com.br/catsim/api/techniques/selection/urry-selector.html)); and
Kullback-Leibler information
([KL](https://douglasrizzo.com.br/catsim/api/techniques/selection/kl-selector.html)), which looks
past the current estimate: it weighs each item over a window of abilities around it, wide at the
start of the test and narrowing as items accumulate. The last line compares the run times of KL and
max info.

In [ ]:
by_selector = {
  "max info": by_estimator["EAP"],
  "Urry": run(selector=UrrySelector()),
  "KL": run(selector=KLSelector()),
}
compare(by_selector)
print(f"KL took {by_selector['KL'].duration / by_selector['max info'].duration:.1f} times as long as max info")

Urry's MSE is 0.079 ± 0.026 above max info's: choosing by difficulty alone ignores the items'
discrimination and guessing, which costs precision on a 3PL bank. It uses more of the bank in
exchange, leaving 22% of it unused against 86%, and two examinees share 24% of their items on
average, against 48%. KL's difference from max info, −0.002 ± 0.010, is within one error of 0, so
this run cannot tell the two apart, and KL's run takes several times as long (the last line).

## Exposure control

Maximum information gives the best items to almost every examinee, who could then pass them on.
Exposure control spreads the items over more of the bank, at some cost in precision. Four methods,
each with the EAP estimator, against max info:

- an exposure cap, `MaxInfoSelector(r_max=R_MAX)`, which passes over an item once it has been given
  to a share `R_MAX` of the examinees tested before;
- [randomesque](https://douglasrizzo.com.br/catsim/api/techniques/selection/randomesque-selector.html)
  selection, which draws each item at random from the five most informative;
- the
  [progressive](https://douglasrizzo.com.br/catsim/api/techniques/selection/progressive-selector.html)
  method, which blends a random score with information, the weight on information growing from 0 at
  the first item to 1 at the last;
- [a-stratification](https://douglasrizzo.com.br/catsim/api/techniques/selection/stratified-selector.html),
  which sorts the bank by discrimination into one stratum per item of the test and gives the strata
  in order, keeping the most discriminating items for the end.

The progressive and a-stratified selectors plan their schedule for a test of known length, so their
`test_size` is `TEST_LENGTH`, the length at which the stopper ends the test. A shorter test would
end before the a-stratified selector reaches its most discriminating stratum, and before the
progressive schedule turns purely informative; a longer one would run past the a-stratified
selector's last stratum, which raises an error.

Under a cap, each session depends on the ones before it, through the exposure rates it sees, so the
cap's Monte Carlo errors, which treat the sessions as independent, are approximations.

In [ ]:
R_MAX = 0.2
by_exposure = {
  "max info": by_estimator["EAP"],
  f"r_max={R_MAX}": run(selector=MaxInfoSelector(r_max=R_MAX)),
  "randomesque": run(selector=RandomesqueSelector(bin_size=5)),
  "progressive": run(selector=ProgressiveSelector(test_size=TEST_LENGTH)),
  "a-stratified": run(selector=AStratSelector(test_size=TEST_LENGTH)),
}
compare(by_exposure)

Each method lowers some exposure metric. The cap holds every item's rate at 0.20 and cuts the
overlap from 0.48 to 0.18. Randomesque and progressive selection lower the top rate to 0.68 and
0.62, and the progressive method leaves only 9% of the bank unused. The MSE costs of these three,
+0.005 ± 0.019, +0.008 ± 0.017 and −0.004 ± 0.021, are each within one error of 0: this run can tell
them neither from no cost nor from one another. a-stratification uses more of the bank than max info
but costs 0.118 ± 0.028 of MSE, four errors: in a 10-item test, half the items come from the less
discriminating half of the bank. Its top rate stays at 1, as for max info: every test starts at the
same estimate, so the least discriminating stratum gives every examinee the same first item.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, name in zip(axes, ["max info", f"r_max={R_MAX}", "a-stratified"], strict=True):
  plot.item_exposure(by_exposure[name].exposure_rates().value, item_bank, par="a", ax=ax, title=name)
plt.tight_layout()
plt.show()

Each point is an item of the bank, at its discrimination, with its exposure rate. Max info gives
only items whose discrimination is above about 1.4, the most discriminating at rates up to 1. The
cap holds those rates at 0.2 and spreads the tests over more of the discriminating items.
a-stratification gives items across the whole range of discrimination, among them the item at rate 1
near $a = 0.8$, the first item of every test.

## Stoppers

The tests so far have a fixed length. A variable-length test ends when a criterion is met, and a
`StopRule` combines several, ending the test when any of them holds
([stopping](https://douglasrizzo.com.br/catsim/api/techniques/stopping/index.html)). Two rules, each
with a cap of `MAX_ITEMS` items:

- precision: the standard error is at most `TARGET_SE`;
- classification: the 95% confidence interval of the estimate lies on one side of the cut score
  `CUT`, so the examinee can be classified as above or below it.

A test that reached the cap may have ended there without meeting the rule's other criterion. The
share classified correctly is the share of examinees whose final estimate is on the same side of
`CUT` as their true ability; the last lines give the classification rule's difference in that share
from the other two designs, paired by examinee. The fixed-length tests of max info are the first
row.

In [ ]:
TARGET_SE = 0.4
CUT = 0.0
MAX_ITEMS = 30
by_stopper = {
  f"{TEST_LENGTH} items": by_estimator["EAP"],
  "precision": run(stopper=StopRule(StandardErrorStopper(TARGET_SE), MaxLengthStopper(MAX_ITEMS))),
  "classification": run(stopper=StopRule(ConfidenceIntervalStopper([CUT]), MaxLengthStopper(MAX_ITEMS))),
}
compare(by_stopper)

correct = {}
for name, result in by_stopper.items():
  examinees = result.examinees()
  correct[name] = ((examinees["theta"] > CUT) == (examinees["true_theta"] > CUT)).astype(float)
  at_cap = numpy.count_nonzero(examinees["test_length"] == MAX_ITEMS)
  print(f"{name}: classified correctly {correct[name].mean():.3f}, tests at the cap {at_cap}")
for name in [f"{TEST_LENGTH} items", "precision"]:
  difference = metrics.mean(correct["classification"] - correct[name])
  print(f"classification minus {name}: {difference.value:+.3f} ± {difference.mc_se:.3f}")

Both variable-length rules give more items than the fixed 10, and so a lower MSE: 0.118 ± 0.017
lower for the precision rule, with 18.6 items on average, and 0.065 ± 0.017 lower for the
classification rule, with 20.0. The precision rule ended 8 of its 500 tests at the 30-item cap. The
classification rule ended 214 there: for examinees near the cut the classification stays in doubt,
and the rule keeps testing them while it stops the others early. It classifies 89.2% of the
examinees correctly, against 85.4% and 88.4% for the other two. Its difference from the fixed-length
test, +0.038 ± 0.016, is more than two errors from 0, but its difference from the precision rule,
+0.008 ± 0.011, is not: this run cannot show that it classifies better than the precision rule,
which gives fewer items. The overlap rate is None for variable-length tests, whose lengths differ.

## Replications, parts and workers

A Monte Carlo error says how much a metric would vary over runs with other seeds. Replications show
it: `N_REPLICATIONS` runs of the EAP design, each with its own seed, give RMSEs whose spread is
close to the error of a single run. A study that needs replications loops over the seeds itself.

A larger study can run on several processes: `parts` splits the examinees, in order, into contiguous
parts, and `workers` runs the parts on that many processes. The result depends on `parts` but never
on `workers`. Without an exposure cap, every examinee's test depends only on their own random
streams, so a run in parts gives the sequential run's tests; with a cap, each part counts exposure
on its own, so its tests differ. On workers, a strategy written in a notebook must move to a module
([04_custom_strategies.ipynb](04_custom_strategies.ipynb)).

In [ ]:
N_REPLICATIONS = 10
rmses = numpy.array([run(rng=seed).rmse().value for seed in range(N_REPLICATIONS)])
print(f"RMSE over {N_REPLICATIONS} replications: {rmses.round(3)}")
print(f"their standard deviation: {rmses.std(ddof=1):.3f}")
print(f"one run's Monte Carlo error: {by_estimator['EAP'].rmse().mc_se:.3f}")

capped = by_exposure[f"r_max={R_MAX}"]
in_parts = run(parts=4, workers=2)
print("no cap: 4 parts on 2 workers give the sequential tests:", in_parts.sessions == by_estimator["EAP"].sessions)
capped_on_two = run(selector=MaxInfoSelector(r_max=R_MAX), parts=4, workers=2)
capped_on_one = run(selector=MaxInfoSelector(r_max=R_MAX), parts=4)
print("cap: 4 parts give the sequential tests:", capped_on_two.sessions == capped.sessions)
print("cap: 4 parts give the same tests on 1 and 2 workers:", capped_on_two.sessions == capped_on_one.sessions)
print(f"cap: RMSE {capped.rmse().value:.3f} in one part, {capped_on_two.rmse().value:.3f} in 4")

The ten RMSEs range from 0.479 to 0.556, and their standard deviation, 0.023, is close to the Monte
Carlo error of the single run, 0.018: one run's errors say how much its metrics would vary over
replications, without running them. With ten replications, that standard deviation is itself only
roughly known.

Run in four parts on two workers, the design without a cap gives exactly the tests of the sequential
run. The capped design gives other tests in four parts, since each part counts exposure from its own
first examinee, but the same tests on one worker as on two; its RMSE barely moves, 0.536 against
0.534.

## Where to go next

- The [Running simulations](https://douglasrizzo.com.br/catsim/running-simulations.html) guide is
  the reference for this notebook's tasks, and the
  [Reproducibility](https://douglasrizzo.com.br/catsim/reproducibility.html) guide for seeds, parts
  and workers. The [technique pages](https://douglasrizzo.com.br/catsim/api/techniques/index.html)
  define each method and say when to choose it.
- [03_a_test_step_by_step.ipynb](03_a_test_step_by_step.ipynb) runs one test a step at a time, as
  `simulate` runs each of them.
- [04_custom_strategies.ipynb](04_custom_strategies.ipynb) writes a strategy of your own.